# Qwen3-4B Heterogeneous-Pool Merge Experiment: One-Click Azure ML Reproduction

Tests arXiv 2511.21437v2's core claim -- "only Task Arithmetic uniquely and
consistently outperforms all source checkpoints when merging n>=6
heterogeneous fine-tunes" -- on **Qwen3-4B**: a base model merged with 7
independently-sourced community fine-tunes (agentic/tool-use, abliterated,
formal theorem proving, Russian instruction-tuning, Chinese grammar
correction, multilingual moral reasoning, behavior simulation) via **five
methods** -- **Linear, TIES, DARE-TIES, Task Arithmetic, Arcee Fusion** --
evaluated on **four benchmarks** -- **ARC-Easy, PIQA, HellaSwag, MMLU**.

This is a different experiment from the MergeKit-paper-repro notebook in
this same repo, not just a rename: different models (Qwen3-4B, not
Llama-2/Meditron), different methods (no LERP/SLERP here; Task Arithmetic
and Arcee Fusion instead), different benchmarks (general-knowledge, not
medical), and a different question being asked (does the merge beat every
source checkpoint, not "does it match a published paper's numbers").

**No HF token needed anywhere** -- none of these 8 checkpoints are gated,
unlike Meditron-7B in the other notebook.

Once merges are done, running the whole thing is one function call:
`run_full_reproduction()`. It submits all 5 merges in parallel, waits,
registers each output as a data asset, submits one combined eval job across
all 13 targets, waits, then downloads and runs the statistical analysis
(McNemar + GEE, matching `analysis.py`'s existing methodology) automatically.

Repo: `alan-turing-institute/model-merging`, folder
`merge-job-qwen3-4b-taskarith/` (this notebook lives there too).


## Prerequisites

Just **Azure CLI with the `ml` extension**, logged into an account with a
role on the `TIRE-1` resource group / `TIRE-2` workspace. No HF token setup
needed for this experiment.


In [ ]:
import os

# Jupyter kernels don't inherit shell customizations from ~/.zshrc/~/.bash_profile,
# so `!az`/subprocess calls to `az` can fail with "command not found" even if
# it works fine in a terminal. az is installed in a venv here rather than on
# the global PATH -- add it once, for this kernel session.
AZ_VENV_BIN = "/Users/mpietrzyk/azure-cli-venv/bin"
if AZ_VENV_BIN not in os.environ["PATH"]:
    os.environ["PATH"] = AZ_VENV_BIN + os.pathsep + os.environ["PATH"]

!az version

## Repo structure

| Category | Files | What it does |
|---|---|---|
| **Merge-only** (used by this notebook) | `job-merge-single-{linear,ties,dare-ties,task-arithmetic}.yml` + `merge_single.sh`; `job-merge-single-arcee-fusion.yml` + `merge_arcee_fusion_only.sh` | Merges exactly one method, writes to a `uri_folder` output, no eval. Arcee Fusion is more involved (7 sequential pairwise stages, 8 mounted inputs) since it only merges 2 models at a time. |
| **Eval** (used by this notebook, combined not split) | `job-eval-single-all.yml` + `eval_all_single.sh` | Evaluates all 13 targets (8 sources + 5 merges) in one job. Unlike the MergeKit-paper-repro notebook, this stays combined: eval loads one model at a time sequentially, so there's no simultaneous multi-model disk/VRAM pressure to split away from. |
| **Combined merge+eval** (legacy, not used by default) | `job.yml` + `run_all.sh` | The original all-in-one run: merges all 5 methods, evaluates all 13 targets, in one job. Succeeded for 4/5 methods historically; Arcee Fusion hit two separate disk-exhaustion bugs in this form (see gotchas). |
| **Not covered by this notebook** | `job-della-n7.yml`, `job-breadcrumbs-n7.yml`, `job-model-stock-n7.yml` | DELLA and Model Breadcrumbs need A100 specifically at n=7 (confirmed hard T4 VRAM ceiling across two separate attempts, at both 4B and 1.5B scale); DELLA also has no working density value in [0.1, 0.5] at n=7 (0.5 collapses like TIES/DARE-TIES did, 0.1 makes it worse, 0.3 confirmed monotonic degradation). Model Stock succeeded with zero tuning but isn't wired into this notebook's one-click flow. Run these manually with the existing job files if needed. |


## Orchestration helpers

Everything below runs `az` via `subprocess`, in your own terminal session
under your own credentials — nothing here is executed by an AI assistant on
your behalf, it's plain Python you're about to run yourself.


In [ ]:
import subprocess
import json
import time

RG = "TIRE-1"
WS = "TIRE-2"

# T4 is the default for every job below -- cheaper, and the .yml files
# themselves already default to it. Pass compute=A100_COMPUTE to any
# run_*/patch_* function for a faster (pricier) one-off run.
DEFAULT_COMPUTE = "azureml:gpu-cluster-merge"  # T4
A100_COMPUTE = "azureml:a100-cluster-merge"

def _az(args):
    """Run an az command, return parsed JSON if possible else raw stdout text."""
    cmd = ["az"] + args
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        raise RuntimeError(f"az command failed: {' '.join(cmd)}\n{result.stderr}")
    try:
        return json.loads(result.stdout)
    except json.JSONDecodeError:
        return result.stdout.strip()

def submit_job(yaml_file, set_args=None, compute=None):
    """Submit a job from a yaml file, optionally with --set key=value pairs
    and/or a compute override (e.g. A100_COMPUTE for a one-off faster run
    without editing the committed T4 default in the yaml file)."""
    args = ["ml", "job", "create", "-f", yaml_file,
            "--resource-group", RG, "--workspace-name", WS]
    all_set_args = dict(set_args or {})
    if compute:
        all_set_args["compute"] = compute
    for k, v in all_set_args.items():
        args += ["--set", f"{k}={v}"]
    args += ["--query", "name", "-o", "tsv"]
    name = _az(args)
    print(f"Submitted {yaml_file} -> {name}" + (f" (compute={compute})" if compute else ""))
    return name

def job_status(name):
    return _az(["ml", "job", "show", "--name", name,
                "--resource-group", RG, "--workspace-name", WS,
                "--query", "status", "-o", "tsv"])

def wait_for_jobs(names_by_label, poll_seconds=30, timeout_seconds=14400):
    """Poll a dict of {label: job_name} until every job reaches a terminal
    state. Returns {label: final_status}."""
    remaining = dict(names_by_label)
    final = {}
    start = time.time()
    while remaining:
        for label, name in list(remaining.items()):
            status = job_status(name)
            if status in ("Completed", "Failed", "Canceled"):
                print(f"  {label} ({name}): {status}")
                final[label] = status
                del remaining[label]
        if remaining:
            if time.time() - start > timeout_seconds:
                raise TimeoutError(f"Timed out waiting for: {list(remaining.keys())}")
            time.sleep(poll_seconds)
    return final

def wait_for_job(name, poll_seconds=30, timeout_seconds=14400):
    """Convenience wrapper around wait_for_jobs for a single job."""
    return wait_for_jobs({"_single": name}, poll_seconds=poll_seconds, timeout_seconds=timeout_seconds)["_single"]

def register_data_asset(name, job_name, output_name, subpath=None):
    """Register a completed job's output (or a subfolder of it, e.g.
    Arcee Fusion's final output sits at merged_model/arcee-fusion-7way
    within its own output) as a new version of a named data asset."""
    tail = f"{job_name}/{output_name}" + (f"/{subpath}" if subpath else "")
    path = f"azureml://datastores/workspaceblobstore/paths/azureml/{tail}"
    result = _az(["ml", "data", "create", "--name", name, "--type", "uri_folder",
                  "--path", path, "--resource-group", RG, "--workspace-name", WS,
                  "-o", "json"])
    version = result["version"]
    print(f"Registered {name} v{version}")
    return version

def download_results(job_name, local_dir):
    subprocess.run(
        ["az", "ml", "job", "download", "--name", job_name,
         "--resource-group", RG, "--workspace-name", WS,
         "--download-path", local_dir, "--output-name", "results"],
        capture_output=True, text=True,
    )

In [ ]:
MERGE_JOBS = {
    "linear": "job-merge-single-linear.yml",
    "ties": "job-merge-single-ties.yml",
    "dare-ties": "job-merge-single-dare-ties.yml",
    "task-arithmetic": "job-merge-single-task-arithmetic.yml",
    "arcee-fusion": "job-merge-single-arcee-fusion.yml",
}

# Arcee Fusion's merge writes its final output to a subfolder within its
# own job output, rather than directly at the output root like the other
# 4 methods -- see merge_arcee_fusion_only.sh.
MERGE_OUTPUT_SUBPATH = {
    "arcee-fusion": "arcee-fusion-7way",
}

EVAL_JOB = "job-eval-single-all.yml"

SOURCES = {
    "base": "Qwen/Qwen3-4B",
    "jan-nano": "Menlo/Jan-nano",
    "abliterated": "mlabonne/Qwen3-4B-abliterated",
    "pythagoras-prover": "Pythagoras-LM/Pythagoras-Prover-4B",
    "qvikhr-instruction": "Vikhrmodels/QVikhr-3-4B-Instruction",
    "chinese-error-corrector": "twnlp/ChineseErrorCorrector4-4B",
    "met-d": "launch/MET-D-Qwen3-4B",
    "osim": "cmu-lti/osim-4b",
}
SOURCE_NAMES_FOR_COMPARISON = [n for n in SOURCES if n != "base"]  # the 7 fine-tunes, not the base itself

METHOD_TO_EVAL_NAME = {
    "linear": "merged-linear", "ties": "merged-ties", "dare-ties": "merged-dare-ties",
    "task-arithmetic": "merged-task-arithmetic", "arcee-fusion": "merged-arcee-fusion",
}

## Results analysis helper

Mirrors `analysis.py`'s existing methodology: for each merge method, (1) does
its accuracy beat every one of the 7 source fine-tunes, and (2) is that
difference statistically real -- McNemar's exact test (merged vs. the single
best-performing source, paired by item) plus a GEE model (binomial, clustered
by item) comparing merged vs. the pooled distribution of all 7 sources.

Extended to run on **ARC-Easy, PIQA, and HellaSwag** (all single flat tasks
with a straightforward `samples_<task>_*.jsonl` per model). **MMLU is
accuracy-only** here, not paired -- lm-eval-harness's aggregate `mmlu` task
spans 57 subject subtasks, each with its own sample file, which needs more
bookkeeping to pair correctly than the other three tasks; add it if you need
that level of rigor for MMLU specifically.


In [ ]:
import glob
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.contingency_tables import mcnemar

PAIRED_TASKS = ["arc_easy", "piqa", "hellaswag"]  # single flat tasks, straightforward to pair by item
ACCURACY_ONLY_TASKS = ["mmlu"]  # aggregate over 57 subtasks -- accuracy only, no item-level pairing here

def load_accuracy(results_dir, model_name, task):
    """Read a model's aggregate accuracy for one task from its results_*.json."""
    files = glob.glob(f"{results_dir}/{model_name}/**/results_*.json", recursive=True)
    if not files:
        return None
    data = json.load(open(sorted(files)[-1]))
    r = data["results"].get(task, {})
    metric = "acc_norm" if task in ("arc_easy", "hellaswag") else "acc"
    return r.get(f"{metric},none", r.get(metric))

def load_samples(results_dir, model_name, task):
    """Read a model's per-item correctness for one task from its samples_*.jsonl."""
    files = glob.glob(f"{results_dir}/{model_name}/**/samples_{task}_*.jsonl", recursive=True)
    if not files:
        return None
    rows = []
    with open(sorted(files)[-1]) as fh:
        for line in fh:
            row = json.loads(line)
            acc = row.get("acc_norm", row.get("acc"))
            rows.append((row["doc_id"], int(acc)))
    return dict(rows)

def analyze_method(results_dir, method, eval_name):
    """Full analysis for one merge method across all tasks: accuracy table,
    beats-all-sources check, McNemar + GEE for the three paired tasks,
    accuracy-only comparison for MMLU."""
    print(f"\n===== {method} ({eval_name}) =====")
    all_tasks = PAIRED_TASKS + ACCURACY_ONLY_TASKS
    for task in all_tasks:
        merged_acc = load_accuracy(results_dir, eval_name, task)
        source_accs = {s: load_accuracy(results_dir, s, task) for s in SOURCE_NAMES_FOR_COMPARISON}
        source_accs = {s: a for s, a in source_accs.items() if a is not None}
        if merged_acc is None or not source_accs:
            print(f"  {task}: MISSING data")
            continue
        best_source = max(source_accs, key=source_accs.get)
        beats_all = all(merged_acc > a for a in source_accs.values())
        print(f"  {task}: merged={merged_acc:.4f}  best_source={best_source}({source_accs[best_source]:.4f})  beats_all_7={beats_all}")

        if task in ACCURACY_ONLY_TASKS:
            continue

        merged_s = load_samples(results_dir, eval_name, task)
        best_s = load_samples(results_dir, best_source, task)
        if merged_s is None or best_s is None:
            print(f"    (no sample-level data for {task}, skipping McNemar/GEE)")
            continue
        common_ids = sorted(set(merged_s) & set(best_s))
        b = sum(1 for i in common_ids if best_s[i] == 1 and merged_s[i] == 0)
        c = sum(1 for i in common_ids if best_s[i] == 0 and merged_s[i] == 1)
        result = mcnemar([[0, b], [c, 0]], exact=True)
        print(f"    McNemar vs best-source ({best_source}): b={b}, c={c}, p={result.pvalue:.4g}")

        long_rows = []
        for s_name in SOURCE_NAMES_FOR_COMPARISON:
            s_samples = load_samples(results_dir, s_name, task)
            if s_samples is None:
                continue
            for item_id, correct in s_samples.items():
                long_rows.append({"item_id": f"{s_name}:{item_id}", "is_merged": 0, "correct": correct})
        for item_id, correct in merged_s.items():
            long_rows.append({"item_id": f"merged:{item_id}", "is_merged": 1, "correct": correct})
        df = pd.DataFrame(long_rows)
        fam = sm.families.Binomial()
        ind = sm.cov_struct.Exchangeable()
        gee = smf.gee("correct ~ is_merged", groups="item_id", data=df, family=fam, cov_struct=ind).fit()
        coef, pval = gee.params["is_merged"], gee.pvalues["is_merged"]
        print(f"    GEE vs pooled-7-sources: coef={coef:.4f}, p={pval:.4g}, OR={np.exp(coef):.4f}")

## One-click reproduction

Submits all 5 merges in parallel (T4 by default), waits, registers each
output as a data asset, submits one combined eval job across all 13
targets (T4 by default), waits, then downloads and runs the statistical
analysis for every method. Pass `compute=A100_COMPUTE` for a faster,
pricier one-off run.


In [ ]:
def run_full_reproduction(compute=DEFAULT_COMPUTE):
    print("=== Step 1/4: submitting merge jobs (5x, parallel) ===")
    merge_names = {m: submit_job(f, compute=compute) for m, f in MERGE_JOBS.items()}

    print("\n=== Step 2/4: waiting for merges ===")
    merge_status = wait_for_jobs(merge_names)
    failed = [m for m, s in merge_status.items() if s != "Completed"]
    if failed:
        raise RuntimeError(f"Merge job(s) did not complete: {failed}")

    print("\n=== Step 3/4: registering merged models as data assets ===")
    for method, name in merge_names.items():
        subpath = MERGE_OUTPUT_SUBPATH.get(method)
        register_data_asset(f"qwen3-merged-{method}-7way", name, "merged_model", subpath=subpath)

    print("\n=== Step 4/4: submitting eval job (all 13 targets, one job) ===")
    eval_name = submit_job(EVAL_JOB, compute=compute)
    eval_status = wait_for_job(eval_name)
    if eval_status != "Completed":
        raise RuntimeError(f"Eval job ended with status {eval_status}")

    local_dir = "results/eval-all"
    download_results(eval_name, local_dir)
    results_dir = f"{local_dir}/named-outputs/results/lm-eval"

    print("\n=== Analysis ===")
    for method, eval_name_key in METHOD_TO_EVAL_NAME.items():
        analyze_method(results_dir, method, eval_name_key)

    return merge_names, eval_name

# One-click full run (T4 by default):
# merge_jobs, eval_job = run_full_reproduction()
#
# Faster, pricier one-off run on A100 instead:
# merge_jobs, eval_job = run_full_reproduction(compute=A100_COMPUTE)
print("Run one of the commented lines above to start.")

## Known gotchas baked into this design

These aren't hypothetical — each one actually happened running this exact
experiment.

**A100's advertised disk doesn't match reality.** The `NC24ads_A100_v4` SKU
used previously had far less usable container disk than its spec sheet
implied. `merge_single.sh`/`merge_arcee_fusion_only.sh` redirect `HF_HOME`
to `/mnt` when writable, since that mount sometimes has more headroom than
the default cache location.

**Arcee Fusion hit two distinct disk-exhaustion bugs before this design.**
First attempt wrote intermediate stage outputs to local disk directly --
exhausted it. Second attempt switched writes to the `rw_mount` output
instead, which still failed with a different error: `rw_mount` writes
*also* buffer through the same local `AZ_BATCH_NODE_ROOT_DIR` quota
(~64GB, invisible to `df -h /`) before flushing to blob, so simply
changing the write target didn't remove the local-disk cost. The actual
fix (used here) is deleting each stage's output as soon as the next stage
has consumed it -- at most 2 stage-sized (~8GB) outputs ever coexist,
regardless of where they're written.

**DELLA/Model Breadcrumbs have a hard T4 VRAM ceiling at n=7** — confirmed
independently at two different model scales (this 4B pool, and a separate
1.5B-scale attempt), with the bottleneck in each case being the
sign-consensus/argsort step needing all 7 models' stacked deltas resident
simultaneously, which scales with n rather than just per-model size. Not a
config issue fixable by an env var — A100 is required. This is why they're
excluded from this notebook's one-click flow rather than silently included
on T4 and left to fail.

**DELLA's density has no working value in [0.1, 0.5] at n=7.** density=0.5
(matching TIES/DARE-TIES's original default) collapses like they did;
density=0.1 (the fix that rescued TIES/DARE-TIES) makes DELLA *worse*, not
better -- attributed to DELLA's stochastic per-row Bernoulli pruning versus
TIES's deterministic top-k selection; density=0.3 confirmed monotonic
degradation across the range, no sweet spot found.

**DARE-TIES's dropout mask is unseeded by default.** Same as the other
notebook in this repo -- every DARE-TIES run is one noisy draw from the
same distribution, not a fixed, reproducible number. Don't read too much
into a single DARE-TIES result without re-running it.
